# 04 — Dataset validation

Validate trajectory records, deduplicate, and build a versioned training dataset with a manifest (hashes, licenses, category counts). Invalid records are reported, never silently dropped.

In [ ]:
import sys
sys.path.insert(0, ".")
from pathlib import Path
print("repo root:", Path.cwd())

In [ ]:
from src.data.dataset_validation import validate_file
summary = validate_file("data/trajectories-v0.jsonl")
print("total:", summary["total_records"], "| valid:", summary["valid_records"], "| invalid:", summary["invalid_records"])
print("categories:", summary["category_counts"])
for err in summary["errors"][:5]:
    print("line", err["line"], err["errors"])

In [ ]:
import json
from src.data.deduplication import dedup_records
records = [json.loads(line) for line in open("data/trajectories-v0.jsonl") if line.strip()]
kept, dropped = dedup_records(records)
print(f"kept {len(kept)}, dropped {len(dropped)}")
for entry in dropped[:5]:
    print("-", entry["reason"], entry["record"]["competition_id"])

In [ ]:
from src.data.dataset_builder import build_dataset
manifest = build_dataset(kept, "data/train.jsonl", "data/manifests/dataset-v0.json")
print("records written:", manifest["record_count"])
print("sha256:", manifest["sha256"])
print("licenses:", manifest["licenses"])
print("categories:", manifest["category_counts"])

In [ ]:
from src.evaluation.leakage_detection import detect_leakage
import json
manifest = json.load(open("evaluation/benchmark_manifest.json"))
held_out = [c["id"] for c in manifest["held_out_competitions"]]
train_ids = sorted({r["competition_id"] for r in kept})
report = detect_leakage(train_ids, held_out)
print("train/eval overlap:", report["overlap_ids"], "| clean:", report["clean"])
assert report["clean"], "training data overlaps the held-out benchmark — fix before training"